# Phase 2 — the r̂ gate (Colab Pro)

Validate our extraction machinery by reproducing **Arditi's refusal direction `r̂`** on
Qwen2.5-7B and checking that the direction *our* code extracts points the same way:
`cosine(our r̂, Arditi's r̂) ≈ 1`.

A near-1 cosine means our activation caching + diff-in-means are trustworthy, so we can believe
the novel consequence direction `v_C` later. Two random directions in 3,584-D sit at |cos|≈0.017,
so ≈1 is unambiguous.

**Before running:** `Runtime → Change runtime type → GPU` (L4 is ideal on Pro). Then run the
cells top to bottom. Step 2 asks you to upload `consequence-awareness.zip`.

> Note: on the pod we run Arditi in its own venv (Rule 1). On Colab we keep the boundary by
> running each heavy step as a **separate subprocess** (our code never imports Arditi's in-process,
> and each model load frees when its process exits).

## 1 · Check the GPU

In [ ]:
import shutil, subprocess, torch
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
if torch.cuda.is_available():
    print("CUDA OK | device:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU on this runtime. Attach one: Colab menu Runtime > Change runtime "
                     "type > GPU (pick L4). In the VS Code Colab extension, reconnect and choose "
                     "a GPU-backed runtime, then re-run.")

## 2 · Install deps + locate the repo
torch is preinstalled on Colab; we add our stack plus the two small libs Arditi's pure-torch
extractor needs (`jaxtyping`, `einops`).

In [ ]:
!pip -q install "transformers>=4.44" accelerate jaxtyping einops
print("deps ready")

**Getting the repo onto the runtime.** The Colab runtime is remote and ephemeral, so
the repo has to arrive somehow. The next cell auto-detects it if it's already present (you cloned
it, synced it via Drive, or the VS Code Colab extension mounted it) and **falls back to a zip
upload** otherwise.

*Optional — persist across disconnects:* uncomment the Drive lines, put the repo under
`/content/drive/MyDrive/`, and it'll be auto-detected (artifacts then survive runtime resets).

In [ ]:
import os, sys, glob, zipfile

# --- how to get the repo here: set ONE of these -------------------------------------------
GIT_URL      = ""     # e.g. "https://github.com/<you>/consequence-awareness.git"  (works everywhere)
ALLOW_UPLOAD = False  # True ONLY in the Colab web UI — the picker widget hangs in VS Code
MOUNT_DRIVE  = False  # True to mount Drive (repo under /content/drive/MyDrive/...)
# ------------------------------------------------------------------------------------------

if MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")

def find_repo():
    p = os.path.abspath(os.getcwd())                     # 1) walk up from cwd
    while p != "/":
        if os.path.exists(os.path.join(p, "configs", "qwen.yaml")):
            return p
        p = os.path.dirname(p)
    for root in ("/content/drive/MyDrive", "/content"):  # 2) common runtime locations
        hits = glob.glob(os.path.join(root, "**", "configs", "qwen.yaml"), recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    return None

REPO = find_repo()

if REPO is None and GIT_URL:                             # 3a) clone (recommended)
    os.system(f"git clone --depth 1 {GIT_URL} /content/consequence-awareness")
    REPO = find_repo()

if REPO is None and ALLOW_UPLOAD:                        # 3b) zip upload (web UI only)
    from google.colab import files
    up = files.upload()                                  # pick consequence-awareness.zip
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content")
    REPO = find_repo()

if REPO is None:                                         # fail FAST, never hang
    raise SystemExit(
        "Repo not on this runtime. Choose one and re-run this cell:\n"
        "  - set GIT_URL to your repo (works in VS Code and the web UI), or\n"
        "  - set MOUNT_DRIVE=True with the repo in your Drive, or\n"
        "  - set ALLOW_UPLOAD=True *only in the Colab web UI* and upload the zip.\n"
        "Note: files.upload() hangs forever in VS Code — its picker needs the Colab web frontend.")

os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))
print("REPO =", REPO)

## 3 · Clone Arditi (reference repo) + build the prompt file
Arditi's harmful/harmless splits ship in-repo; `phase2_build_refusal.py` copies them verbatim
into `data/contrast/refusal.jsonl` (both sides of the gate use these same prompts).

In [ ]:
import os, subprocess
ARDITI = os.path.join(REPO, "external", "refusal_direction")
if not os.path.exists(ARDITI):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/andyrdt/refusal_direction", ARDITI], check=True)
print("splits:", os.listdir(os.path.join(ARDITI, "dataset", "splits")))
!python scripts/phase2_build_refusal.py
!head -1 data/contrast/refusal.jsonl

## 4 · Cache activations with OUR code (GPU)
Runs stage 01 as a subprocess. **First run downloads Qwen2.5-7B (~15 GB) — a few minutes.**
Writes `artifacts/activations/refusal_Qwen2.5-7B-Instruct.npz`.

In [ ]:
!python scripts/01_cache_acts.py --dataset refusal
!ls -la artifacts/activations/

## 5 · Run Arditi's extraction on the SAME prompts (GPU)
Separate subprocess (memory frees after step 4; Arditi runs process-isolated). Reuses the model
already in the HF cache, so no second download. Writes the diff-in-means cube
`artifacts/directions/r_hat_mean_diffs.pt` = `[n_positions, n_layers, d_model]`.

In [ ]:
# subprocess (not %%bash) so a failure RAISES here instead of scrolling past as stderr.
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH=".")
p = subprocess.run(
    [sys.executable, "../../scripts/arditi_qwen25/run_extract.py",
     "--refusal", "../../data/contrast/refusal.jsonl",
     "--model",   "Qwen/Qwen2.5-7B-Instruct",
     "--out",     "../../artifacts/directions/r_hat_mean_diffs"],
    cwd=os.path.join(REPO, "external", "refusal_direction"), env=env,
    text=True, capture_output=True)
print(p.stdout[-4000:])
if p.returncode != 0:
    print("STDERR:\n", p.stderr[-4000:])
    raise SystemExit(f"Arditi extraction failed (exit {p.returncode}) — see STDERR above")
print("OK ->", os.path.join(REPO, "artifacts/directions/r_hat_mean_diffs.pt"))

## 6 · The gate (CPU)
Compute OUR r̂ per layer (diff-in-means at the last prompt token) and compare to Arditi's cube's
last-position slice. The two codebases index layers off-by-one (our stored `k` = block-*output*
`h[k+1]`; Arditi's layer `l` = block-*input* `h[l]`), so we sweep offsets −1/0/+1 — the one that
lights up to ≈1 confirms the alignment **and is the gate**.

In [ ]:
import os, numpy as np, torch, glob
import matplotlib.pyplot as plt

hits = glob.glob("artifacts/activations/refusal_*.npz")
assert hits, "no refusal activation cache — run step 4 first"
assert os.path.exists("artifacts/directions/r_hat_mean_diffs.pt"), \
    "no Arditi cube — run step 5 first"

z = np.load(hits[0], allow_pickle=True)
acts, labels = z["acts"], z["labels"]                      # [n, n_layers, d]; index k = h[k+1]
print(f"our cache: acts{acts.shape}  ({int((labels==1).sum())} harmful / "
      f"{int((labels==0).sum())} harmless)")
our = np.stack([acts[labels==1,k,:].mean(0) - acts[labels==0,k,:].mean(0)
                for k in range(acts.shape[1])])
our = torch.tensor(our, dtype=torch.float32); our /= our.norm(dim=-1, keepdim=True)

cube = torch.load("artifacts/directions/r_hat_mean_diffs.pt", map_location="cpu").float()
ard = cube[-1]                                             # last position = final prompt token
nrm = ard.norm(dim=-1, keepdim=True)
DEAD = (nrm.squeeze(-1) < 1e-8)                            # zero-norm layers -> would give nan
ard = ard / nrm.clamp_min(1e-12)                           # [n_layers, d]; layer l = h[l]
if DEAD.any():
    print("zero-norm Arditi layers (excluded):", DEAD.nonzero().flatten().tolist(),
          "\n  -> expected at layer 0: it is the raw EMBEDDING, and every prompt ends with the"
          "\n     same chat-template token, so mean(harmful)-mean(harmless) is exactly 0 there.")

n = our.shape[0]
def cos_at(off):                                           # skip dead layers, never return nan
    return np.array([float(torch.dot(our[k], ard[k+off]))
                     for k in range(n) if 0 <= k+off < n and not DEAD[k+off]])
for off in (-1, 0, 1):
    c = cos_at(off)
    print(f"offset {off:+d}: mean|cos|={np.abs(c).mean():.4f}  max={c.max():.4f}  (n={len(c)})")
print("\nper-layer at offset +1:", np.round(cos_at(1), 4).tolist())

# random-direction noise floor for scale
g = torch.Generator().manual_seed(0)
R = torch.randn(1000, our.shape[1], generator=g); R /= R.norm(dim=-1, keepdim=True)
null_p95 = float((R @ our[n//2]).abs().quantile(0.95))
print("random |cos| p95 (noise floor):", round(null_p95, 3))

In [ ]:
c = cos_at(1)                                              # +1 is the expected alignment (dead layers skipped)
plt.figure(figsize=(8,4))
plt.axhspan(-null_p95, null_p95, color="gray", alpha=.2, label="random null band")
plt.plot(range(len(c)), c, "o-", label="cos(our r̂, Arditi r̂)")
plt.axhline(1, ls="--", color="green"); plt.ylim(-.1, 1.05)
plt.xlabel("layer"); plt.ylabel("cosine"); plt.title("r̂ gate — our extraction vs Arditi")
plt.legend(); plt.grid(alpha=.3); plt.show()
print("GATE:", "PASS ✅  (cos≈1 across mid layers)" if np.median(c) > 0.9
      else "INVESTIGATE ⚠️  (a bug in caching / token position / layer indexing)")

## 6b · Red-team the gate (run this before believing the number)

A high cosine is only meaningful if it *could* have come out low. These four checks separate
"our machinery is right" from "this comparison can't fail":

1. **Cross-layer matrix** — does offset +1 actually beat other offsets, or is everything high?
2. **Self-similarity** — are adjacent layers of our own cube already ~1.0? If so the offset
   test has no discriminating power and proves nothing about alignment.
3. **Label permutation (the decisive one)** — shuffle harmful/harmless, recompute, re-compare.
   A sound pipeline must COLLAPSE to the noise floor. If it stays high, the agreement never
   depended on the labels and the gate is an artifact.
4. **Outlier dominance** — is the direction just a few huge residual-stream dimensions?

In [ ]:
import numpy as np, torch, glob
z = np.load(glob.glob("artifacts/activations/refusal_*.npz")[0], allow_pickle=True)
acts, labels = z["acts"], z["labels"]
cube = torch.load("artifacts/directions/r_hat_mean_diffs.pt", map_location="cpu").float()
nrm = ard_n = cube[-1].norm(dim=-1, keepdim=True)
DEAD = (nrm.squeeze(-1) < 1e-8)          # layer 0 is the raw embedding -> exactly zero
ard = cube[-1] / nrm.clamp_min(1e-12)
LIVE = [k for k in range(cube.shape[1]-1) if not DEAD[k+1]]   # k usable at offset +1

def dirs(lbl):
    v = np.stack([acts[lbl==1,k,:].mean(0) - acts[lbl==0,k,:].mean(0) for k in range(acts.shape[1])])
    t = torch.tensor(v, dtype=torch.float32)
    return t / t.norm(dim=-1, keepdim=True)

our = dirs(labels); n = our.shape[0]
M = our @ ard.T                     # M[k, l] = cos(our[k], ard[l])

print("1) CROSS-LAYER  (want +1 clearly highest)")
for off in (-1, 0, 1, 2):
    vals = [M[k, k+off] for k in range(n) if 0 <= k+off < n and not DEAD[k+off]]
    print(f"     offset {off:+d}: mean cos = {torch.tensor(vals).mean():.4f}")
far = torch.tensor([M[k, l] for k in range(n) for l in range(n)
                    if abs((k+1)-l) > 3 and not DEAD[l]])
print(f"     far-apart layers (|Δ|>3): mean |cos| = {far.abs().mean():.4f}   <-- if this is also"
      " high, the test does NOT discriminate")

S = our @ our.T
print(f"\n2) SELF-SIMILARITY  our[k] vs our[k+1]: mean cos = {S.diagonal(1).mean():.4f}"
      "   <-- ~1.0 means adjacent layers are indistinguishable")

# 3) PERMUTATION NULL — the decisive check. One shuffle is NOT enough: the null is wide
#    (sd ~0.4), because a random split's accidental class imbalance projects onto a
#    high-variance axis. Compare the true value against the DISTRIBUTION, not a threshold.
def score(lbl):
    Mx = dirs(lbl) @ ard.T
    return float(torch.tensor([Mx[k, k+1] for k in LIVE]).mean())

true_s = score(labels)
rng = np.random.default_rng(0)
null = np.array([score(np.random.default_rng(i).permutation(labels)) for i in range(200)])
pval = float((null >= true_s).mean())
print(f"
3) PERMUTATION NULL (200 shuffles)")
print(f"     true = {true_s:.4f} | null mean {null.mean():.4f} sd {null.std():.4f} "
      f"max {null.max():.4f} | p = {pval:.4f}")

mid = our[n//2].abs()
top = mid.topk(10).values
print(f"\n4) OUTLIER DOMINANCE  top-10 of 3584 dims hold "
      f"{(top.pow(2).sum()/mid.pow(2).sum()):.1%} of the direction   <-- >50% = dominated")

g = torch.Generator().manual_seed(0)
R = torch.randn(1000, our.shape[1], generator=g); R /= R.norm(dim=-1, keepdim=True)
print(f"\n   noise floor |cos| p95 = {float((R @ our[n//2]).abs().quantile(0.95)):.4f}")

ok = (true_s > 0.9) and (pval < 0.01)
print("\nVERDICT:", "gate is REAL ✅ (true labels beat every permutation)" if ok
      else "SUSPICIOUS ⚠️ — permutations reach the true value, or offset +1 is not high.")

## 7 · (Optional) cache the consequence set for Phase 3
While the model is warm, cache the v_C activations too. Only trust these once the gate above
passes. Then Phase 3 (extract v_C, probe held-out framings) runs entirely on CPU.

In [ ]:
!python scripts/01_cache_acts.py --dataset consequence
!ls -la artifacts/activations/